In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

columnname=StructType([StructField('custid', StringType(), True), StructField('fname', StringType(), True), StructField('lname', StringType(), True), StructField('age', StringType(), True), StructField('profession', StringType(), True)])

leftdf=spark.read.schema(columnname).csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB2 Customers/custsmodified").where("custid<>'ten' and custid<>'trailer_data:end of file'").na.drop().withColumn("age",regexp_replace("age","[^0-9]",""))
#leftdf.show(10)

rightdf=leftdf.where("custid in (4000001,4000002,4000003,4000011,4000012,4000013)")
#rightdf.show(10)

#Inner Join
innerjoindf=leftdf.join(rightdf,how='inner',on='custid').orderBy("custid")
#innerjoindf.show(30)

#leftouter Join / Rightouter Join
leftjoindf=leftdf.join(rightdf,how='left',on='custid').orderBy("custid")
#leftjoindf.show(100)

#Full Join
fulljoindf=leftdf.join(rightdf,how='full',on='custid').orderBy("custid")
#fulljoindf.show(100)

#Semi and Anti (Special Join)
semijoindf=leftdf.join(rightdf,how='semi',on='custid').orderBy("custid") #how='anti'
semijoindf.show(100)


In [0]:
txn1_2025=spark.read.csv("/Volumes/telecom_catalog_assign/landing_zone/landing_vol/BB2 Customers/txns_2025.txt",inferSchema=True).toDF("txnid","txndate","custid","amount","category","product","city","state","txntype")
#txn1_2025.show(10)

txnmungeddf=txn1_2025.na.drop().dropDuplicates(["txnid"])
txnenrichdf=txnmungeddf.withColumn("sk",monotonically_increasing_id())
#print(txnenrichdf.count())

txndecdf=txnenrichdf.filter("month(txndate)=12")
#print(txndecdf.count())

custdf=leftdf.select("custid","fname","lname","age","profession").na.drop()
joineddf=custdf.alias("c").join(txndecdf.alias('t'),how='inner',on='custid').orderBy("custid")
#top 3 customers
#display(joineddf.select("c.custid","c.fname","c.lname","t.txndate","t.amount","t.category","t.product","t.sk").orderBy("amount",ascending=False).limit(3))

#Least 3 customers
#display(joineddf.select("c.custid","c.fname","c.lname","t.txndate","t.amount","t.category","t.product","t.sk").orderBy("amount",ascending=True).limit(3))

#semi and anti (for lookup purpose), for lookup inner or left is not preferred.
semijoindf=custdf.join(txndecdf,how="semi",on="custid") 
#display(semijoindf.count())
print("")
newinnerjoindf=custdf.join(txndecdf,how="inner",on="custid").orderBy("custid")
#display(newinnerjoindf.count())
latestinnerjoindf=newinnerjoindf.select("custid","fname","lname","age","profession","txnid","txndate","amount","category","product","city","state","txntype")
#display(latestinnerjoindf)

#anti
antijoindf=custdf.join(txndecdf,how="anti",on="custid")
#display(antijoindf.count())

from pyspark.sql.window import Window
window1=latestinnerjoindf.withColumn("seqno",row_number().over(Window.partitionBy("custid","txndate").orderBy(desc("amount")))).withColumn("denserank",rank().over(Window.partitionBy("custid").orderBy("seqno")))
display(window1)

window1=latestinnerjoindf.withColumn("seqno",row_number().over(Window.partitionBy("custid","txndate").orderBy(desc("amount")))).withColumn("denserank",dense_rank().over(Window.partitionBy("custid").orderBy("seqno")))
display(window1)
